# Lensless Imaging — Demo

This notebook runs our lensless reconstruction end to end on any dataset you provide

**How to use:**

1. Fill in `DATA_URL` in the configuration cell below
2. Run all cells top to bottom (`Runtime -> Run all`).
3. The notebook downloads the checkpoint and the dataset, runs `inference.py`, visualizes samples, and prints the metrics

Designed to work in a Google Colab

## 1. Configuration

- `MODEL` — which model to run (its checkpoint is downloaded below)
- `DATA_URL` — Google Drive link to a `.zip` dataset (structure shown in step 5)
- `SAVE_PATH` — where reconstructions are saved (default: `reconstructions`)

The checkpoint Google Drive ids live in `download_checkpoints.py`

In [ ]:
REPO_URL = "https://github.com/danyatennn/Lensless_Imaging"
MODEL = "modular_pre_post"  # modular_pre_post | modular_pre | modular_post | unrolled_admm20
DATA_URL = "https://drive.google.com/file/d/1Z2Pos-4maZDeidN3BfhNZhDdaM8rmyyF/view?usp=sharing"
SAVE_PATH = "reconstructions"

## 2. Clone the repository

In [ ]:
import os, sys, subprocess

if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
if os.path.basename(os.getcwd()) != "repo":
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python download_checkpoints.py --model $MODEL

## 5. Download and unzip the dataset

Paste a Google Drive link to a `.zip` in `DATA_URL` above. The archive must unzip to a directory of the form:

```
data
├── lensless        # lensless measurements (required)
│   └── ID.png
├── masks           # mask patterns (required)
│   └── ID.npy
└── lensed          # ground-truth images (optional)
    └── ID.png
```

In [ ]:
import zipfile
from pathlib import Path

import gdown

if not os.path.exists("dataset.zip"):
    gdown.download(DATA_URL, "dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("dataset.zip") as archive:
    archive.extractall("demo_data")

DATA_DIR = next(p.parent for p in Path("demo_data").rglob("lensless") if p.is_dir())
RECON_DIR = Path(SAVE_PATH).absolute()
print("data dir:", DATA_DIR)
print("reconstructions ->", RECON_DIR)

## 6. Run inference

Runs `inference.py` on the custom directory and saves id-matched reconstructions to `RECON_DIR`.

In [ ]:
!python inference.py \
    model=$MODEL \
    inferencer.from_pretrained=saved/$MODEL/model_best.pth \
    datasets=custom_dir \
    datasets.test.data_dir="$DATA_DIR" \
    inferencer.save_path="$RECON_DIR"

## 7. Visualize some samples

Original (if available) vs. lensless measurement vs. reconstruction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

recon_paths = sorted(RECON_DIR.glob("*.png"))[:4]
lensless_dir = DATA_DIR / "lensless"
lensed_dir = DATA_DIR / "lensed"

n = len(recon_paths)
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
axes = np.atleast_2d(axes)
for row, recon_path in enumerate(recon_paths):
    sample_id = recon_path.stem
    lensless = np.asarray(
        Image.open(lensless_dir / f"{sample_id}.png").convert("RGB")
    ).astype(np.float32)
    lensless = lensless / (lensless.max() + 1e-8)
    reconstruction = np.asarray(Image.open(recon_path).convert("RGB"))

    lensed_path = lensed_dir / f"{sample_id}.png"
    if lensed_path.exists():
        axes[row, 0].imshow(Image.open(lensed_path).convert("RGB"))
    axes[row, 0].set_title("original (lensed)")
    axes[row, 1].imshow(lensless)
    axes[row, 1].set_title("lensless measurement")
    axes[row, 2].imshow(reconstruction)
    axes[row, 2].set_title("reconstruction")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Compute metrics

If the dataset has ground-truth `lensed` images, print PSNR / SSIM / LPIPS / MSE.

In [ ]:
!python calculate_metrics.py \
    --ground_truth "$DATA_DIR" \
    --reconstructions "$RECON_DIR"